<a href="https://colab.research.google.com/github/LeonardooAlves/LSE-ST445/blob/main/Week%207/1_st445_week7_lab1_plotting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ST445 Week 7 · Lab 1: plotting with Matplotlib and seaborn

**About 90 minutes · self-guided · follows Part 2 of the Week 7 lecture**

This lab teaches the mechanics of making charts in Python, so that Lab 2 can concentrate on exploring a real dataset. By the end you will be able to:

1. explain how a Matplotlib chart is organised, and use the **`fig, ax`** convention;
2. draw the common chart types with **Matplotlib** and change their details (labels, colours, reference lines, annotations);
3. draw the same charts, and several more, with **seaborn**, and combine the two libraries in one figure;
4. **reproduce eight target charts** from a picture and a short description.

All the data are **synthetic**: generated by code with a fixed random seed, so everyone sees identical charts.

**Symbols used on code cells:** 🟢 run the cell and read the explanation · 🔵 write the code yourself. Each exercise is followed by a check cell that compares your chart's structure (the number of plots, the titles and the axis labels) with the target.

**Before you start:** choose **File → Save a copy in Drive**, so that your changes are saved.

### Documentation you will use

| Library | Start here | Reference for every function |
|---|---|---|
| Matplotlib | [Quick start guide](https://matplotlib.org/stable/users/explain/quick_start.html) | [API reference](https://matplotlib.org/stable/api/index.html) |
| seaborn | [Overview of plotting functions](https://seaborn.pydata.org/tutorial/function_overview.html) | [API reference](https://seaborn.pydata.org/api.html) |
| pandas | [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) | [API reference](https://pandas.pydata.org/docs/reference/index.html) |

Every function introduced below links to its page in the reference, where all its parameters are listed.

---
## 0. Set-up

The first cell **imports** the libraries (makes them available in the notebook), defines a colour-blind-safe palette, and changes a few of Matplotlib's default settings.

**What `plt.rcParams.update` does.** Matplotlib stores its default settings (font size, line width, which borders to draw, and hundreds more) in a dictionary called [`rcParams`](https://matplotlib.org/stable/users/explain/customizing.html). Updating it once at the start applies the new defaults to every figure drawn afterwards, so we do not have to repeat them in each chart. Removing the top and right borders ("spines") is a small example of the lecture's data-ink principle: they carry no information.

In [ ]:
# 🟢 Imports, a colour palette and default settings
import numpy as np                 # numerical arrays and random numbers
import pandas as pd                # tables (DataFrames)
import matplotlib.pyplot as plt    # Matplotlib's plotting interface, by convention called plt
import seaborn as sns              # statistical charts built on Matplotlib, by convention called sns

# A colour-blind-safe palette (Okabe and Ito, 2008): eight colours that stay distinct
# for people with the common forms of colour vision deficiency
OKABE_ITO = [
    "#0072B2",  # blue
    "#E69F00",  # orange
    "#009E73",  # bluish green
    "#CC79A7",  # reddish purple
    "#56B4E9",  # sky blue
    "#D55E00",  # vermilion
    "#F0E442",  # yellow
    "#000000",  # black
]

# Default settings applied to every figure in this notebook
plt.rcParams.update({
    "figure.dpi": 110,             # resolution on screen, in dots (pixels) per inch
    "axes.spines.top": False,      # remove the top border line of every plot
    "axes.spines.right": False,    # remove the right border line of every plot
    "font.size": 11,               # default text size, in points
})

### The data

Two small synthetic datasets:

* **`students`**: 300 students, with their programme, year of study, weekly study hours, class attendance, hours of sleep and exam score (out of 100);
* **`sales`**: monthly sales, in thousands of pounds, for three branches of a shop (North, South and East), from January 2022 to December 2024.

The cell below generates them. You do not need to follow every line, but the comments explain what each one does.

In [ ]:
# 🟢 Generate the two datasets
rng = np.random.default_rng(445)      # a random-number generator with a fixed seed:
                                      # everyone running this cell gets identical data

# ---- Dataset 1: 300 students -------------------------------------------------
n = 300                                                        # number of students

programme = rng.choice(["Data Science", "Economics", "Statistics"],   # possible values
                       size=n,                                 # one value per student
                       p=[0.4, 0.35, 0.25])                    # the share of each programme
year = rng.choice(["Year 1", "Year 2"], size=n)                # year of study
hours = rng.gamma(shape=4, scale=2.5, size=n).round(1)         # weekly study hours: a gamma distribution
                                                               # gives positive, right-skewed values
attendance = rng.uniform(55, 100, size=n).round(0)             # % of classes attended, evenly spread
sleep = rng.normal(7, 1, size=n).round(1)                      # hours of sleep a night: mean 7, sd 1

# Each programme adds a small amount to the score, so that the groups differ
bonus = pd.Series(programme).map({"Data Science": 7, "Economics": 0, "Statistics": 3.5}).to_numpy()
score = 25 + 2.2 * hours + 0.25 * attendance + bonus + rng.normal(0, 7, size=n)   # plus random noise

students = pd.DataFrame({
    "programme": programme,
    "year": year,
    "study_hours": hours,
    "attendance_pct": attendance,
    "sleep_hours": sleep,
    "exam_score": score.clip(0, 100).round(0),                 # keep scores between 0 and 100
})

# ---- Dataset 2: monthly sales for three branches, January 2022 to December 2024 -
months = pd.date_range("2022-01-01", "2024-12-01", freq="MS")  # "MS": the first day of each month
season = np.sin(2 * np.pi * (months.month - 3) / 12)           # a yearly wave, between -1 and 1

rows = []
for branch, base, growth in [("North", 120, 1.5), ("South", 100, 2.6), ("East", 140, 0.4)]:
    # sales = starting level + steady growth + seasonal wave + random noise
    sales = base + growth * np.arange(len(months)) + 12 * season + rng.normal(0, 4, len(months))
    for month, value in zip(months, sales):
        rows.append({"month": month, "branch": branch, "sales_k": round(value, 1)})
sales = pd.DataFrame(rows)

print("students:", students.shape, "| sales:", sales.shape)

In [ ]:
# 🟢 The first five rows of each table
students.head()

In [ ]:
sales.head()

**What you should see:** `students: (300, 6) | sales: (108, 3)`, meaning 300 rows and 6 columns, and 108 rows and 3 columns, followed by the first five rows of each table.

`sales` is in **long format**: each row holds one branch in one month, so the three branches are stacked in one column, `branch`, rather than spread across three columns. 3 branches × 36 months = 108 rows. seaborn works best with data in this format (section 3).

---
## 1. How a Matplotlib chart is organised: `fig` and `ax`

### The figure and the axes

The **`fig`** object, or **figure**, is the entire "canvas" on which everything is drawn. Think of it as the main container that holds all parts of your chart. A figure can include several plots, titles, annotations and legends that apply across all of them. A figure can contain any number of **axes** objects (individual plots), but it usually has at least one.

The **`ax`** object, or **axes**, is one individual plot within the figure. When we think of a "plot", complete with x- and y-axis labels, tick marks, data and gridlines, that is an axes object. If a figure holds several plots, each one is a separate axes object within the same figure. The axes is the object you will use most often: it is where you plot data, and where you set a plot's title, axis labels, limits and ticks.

Using both `fig` and `ax` gives you precise control, especially as charts become more complex: you can manage the layout of several plots, customise each plot separately, and add elements that belong to the whole figure, such as an overall title or a colour bar. For a quick look at the data you can skip `fig` and `ax` and call functions such as `plt.plot(...)` and `plt.title(...)` directly. That works for simple cases, but as your charts grow, using `fig` and `ax` is the recommended practice ([Matplotlib: the two interfaces](https://matplotlib.org/stable/users/explain/figure/api_interfaces.html)).

> **"Axes" and "axis" are different things.** An **axes** is a whole plot. An **axis** is one of its two number lines, the x-axis or the y-axis. Each axes contains two axis objects.

The [anatomy of a figure](https://matplotlib.org/stable/gallery/showcase/anatomy.html) page on the Matplotlib website labels every part of a chart with its name.

### The four-step pattern

```python
fig, ax = plt.subplots()      # 1. create a figure that contains one axes
ax.plot(x, y)                 # 2. draw the data on the axes
ax.set_title("...")           # 3. label the axes: title and axis labels
ax.set_xlabel("...")
ax.set_ylabel("...")
plt.show()                    # 4. display the figure (or fig.savefig("file.png") to save it)
```

[`plt.subplots()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html) **returns two objects**: the figure and the axes. Writing `fig, ax = ...` stores the first in the variable `fig` and the second in `ax` (Python calls this **unpacking**).

**Keyword arguments.** Most functions accept optional settings written as `name=value` inside the brackets, for example `figsize=(8, 3.5)`. These are **keyword arguments**: you may give them in any order, and any you leave out keep their default value.

In [ ]:
# 🟢 One figure containing one axes: the basic pattern
north = sales[sales["branch"] == "North"]          # keep only the North branch's 36 rows

fig, ax = plt.subplots(figsize=(8, 3.5))           # figsize=(width, height), in inches
ax.plot(north["month"], north["sales_k"])          # x values, then y values: draws a line
ax.set_title("Monthly sales, North branch")        # title of this plot
ax.set_xlabel("month")                             # label of the x-axis
ax.set_ylabel("sales (£ thousand)")                # label of the y-axis, with its unit
plt.show()                                         # display the figure below the cell

**What you should see:** one line that rises over three years, with a regular yearly wave (the seasonal pattern built into the data). The title and both labels were set **on the axes**, `ax`, because they belong to that particular plot.

**[`ax.plot(x, y)`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.plot.html)** draws a line through the points (x, y) in order. Here x is a column of dates and y a column of sales values; Matplotlib formats the dates on the x-axis automatically.

### Setting several properties at once: `ax.set`

[`ax.set(...)`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.set.html) sets several properties of the axes in one call. Each keyword argument names a property: `title=` does the same as `ax.set_title(...)`, `xlabel=` the same as `ax.set_xlabel(...)`, and so on. It keeps the code short and easy to read.

In [ ]:
# 🟢 The same chart, with the labels set in one call and the line styled
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(north["month"], north["sales_k"],
        color=OKABE_ITO[0],         # the line colour: the first colour of the palette (blue)
        linewidth=2)                # the line thickness, in points (the default is 1.5)
ax.set(title="Monthly sales, North branch",
       xlabel="month",
       ylabel="sales (£ thousand)")
plt.show()

### The shorter `plt.` interface

Matplotlib also has a set of functions in `plt` itself: [`plt.plot`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html), [`plt.title`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.title.html), [`plt.xlabel`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.xlabel.html) and many others with matching names. Each of these functions acts on the **current axes**: the plot that Matplotlib has most recently created or used, which it tracks for you behind the scenes. This is convenient for a single quick chart. With several plots in one figure, however, it is no longer obvious which plot a command will change, and that is when the explicit `fig, ax` style becomes clearer.

In [ ]:
# 🟢 The same chart with the plt. functions
plt.figure(figsize=(8, 3.5))                        # create a figure; its axes is created by plt.plot
plt.plot(north["month"], north["sales_k"])           # draws on the current axes
plt.title("Monthly sales, North branch (plt. interface)")
plt.ylabel("sales (£ thousand)")
plt.show()

### Several plots in one figure

`plt.subplots(nrows, ncols)` creates a figure with a **grid** of axes, and returns the figure and an **array** (a NumPy array) of axes objects. You choose which plot to draw on by indexing the array:

* with one row, `axs[0]` is the first plot and `axs[1]` the second;
* with several rows and columns, use two indices, `axs[row, column]`: `axs[0, 1]` is the top-right plot of a 2 × 2 grid. Counting starts at 0.

By convention, the array is called `axs` (plural) when it holds more than one axes.

Some settings belong to the **whole figure** rather than to one plot:

* [`fig.suptitle("...")`](https://matplotlib.org/stable/api/_as_gen/matplotlib.figure.Figure.suptitle.html) adds one title above all the plots ("suptitle" stands for "super title").
* **`layout="constrained"`**, given to `plt.subplots`, switches on Matplotlib's [constrained layout](https://matplotlib.org/stable/users/explain/axes/constrainedlayout_guide.html). It automatically adjusts the size and spacing of the plots so that titles, axis labels, tick labels and legends do **not overlap** one another or get cut off at the edge of the figure. Without it, plots are placed on a fixed grid, and long labels often collide.

In [ ]:
# 🟢 One figure, two plots side by side
fig, axs = plt.subplots(1, 2,                       # 1 row, 2 columns of plots
                        figsize=(11, 3.5),
                        layout="constrained")       # space the plots so nothing overlaps

axs[0].plot(north["month"], north["sales_k"], color=OKABE_ITO[0])
axs[0].set(title="North: over time", xlabel="month", ylabel="sales (£ thousand)")

axs[1].hist(north["sales_k"], bins=12, color=OKABE_ITO[1])       # a histogram with 12 bars
axs[1].set(title="North: distribution of monthly sales", xlabel="sales (£ thousand)", ylabel="number of months")

fig.suptitle("Two views of the same data", fontweight="bold")     # one title for the whole figure
plt.show()

**What you should see:** two plots, with one overall title above them. The left plot shows sales over time; the right plot shows how many months fell into each range of sales values (a histogram, explained in section 2.4).

Try it: delete `layout="constrained"` from the cell and run it again. The plots move closer together and the labels may overlap.

### Sharing an axis between plots

`sharey=True` gives all the plots in the grid **the same y-axis range**, and `sharex=True` the same x-axis range. When plots are meant to be compared, sharing the scale makes the comparison fair: without it, each plot chooses its own range, and a small change can look as large as a big one. This is the lecture's **small multiples** principle: the same chart, repeated, on the same scale.

In [ ]:
# 🟢 Three plots in a row, one per branch, sharing the same y-axis range
fig, axs = plt.subplots(1, 3, figsize=(13, 3.5),
                        sharey=True,                # all three plots use the same y-axis range
                        layout="constrained")

branches = ["North", "South", "East"]
for i, branch in enumerate(branches):               # i = 0, 1, 2 and branch = "North", "South", "East"
    data = sales[sales["branch"] == branch]         # the rows for this branch
    axs[i].plot(data["month"], data["sales_k"], color=OKABE_ITO[i])
    axs[i].set_title(branch)
    axs[i].tick_params(axis="x", rotation=45)       # rotate the x-axis tick labels by 45 degrees

axs[0].set_ylabel("sales (£ thousand)")             # the axis is shared, so one label is enough
fig.suptitle("Monthly sales by branch, 2022–2024", fontweight="bold")
plt.show()

**What you should see:** three plots on the same vertical scale. South starts lowest but grows fastest; East hardly changes. Because the scale is shared, these differences can be read directly.

**How the loop works.** [`enumerate`](https://docs.python.org/3/library/functions.html#enumerate) goes through the list `branches` and gives, at each step, the position `i` (0, 1, 2) together with the item (`"North"`, `"South"`, `"East"`). The position selects both the plot, `axs[i]`, and its colour, `OKABE_ITO[i]`.

**[`ax.tick_params`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.tick_params.html)** changes the appearance of the tick marks and their labels; here it rotates the date labels so they do not run into each other.

### Saving a figure

[`fig.savefig("name.png")`](https://matplotlib.org/stable/api/_as_gen/matplotlib.figure.Figure.savefig.html) writes the figure to a file. Saving belongs to the **figure**, because it is the whole canvas that is saved. Two keyword arguments are worth always using:

* **`dpi=200`** sets the resolution in dots (pixels) per inch: higher values give sharper images for reports and slides;
* **`bbox_inches="tight"`** trims the empty margin around the figure and makes sure no label is cut off.

The file format follows the file name's extension: `.png`, `.pdf`, `.svg` and others.

In [ ]:
# 🟢 Save the last figure as a PNG file (open the Files panel on the left to see it)
fig.savefig("sales_by_branch.png", dpi=200, bbox_inches="tight")
print("Saved sales_by_branch.png")

---
## 2. Common charts in Matplotlib

As in the lecture, each chart is chosen for the **question** it answers. Each drawing method belongs to the axes:

| Question | Chart | Matplotlib method |
|---|---|---|
| How does a value change over time? | line chart | [`ax.plot`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.plot.html) |
| How are two numeric variables related? | scatter plot | [`ax.scatter`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.scatter.html) |
| How do categories compare? | bar chart | [`ax.bar`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.bar.html) (vertical), [`ax.barh`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.barh.html) (horizontal) |
| How is one numeric variable distributed? | histogram, box plot | [`ax.hist`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.hist.html), [`ax.boxplot`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.boxplot.html) |

The [Matplotlib gallery](https://matplotlib.org/stable/gallery/index.html) shows hundreds of examples, each with its code.

### 2.1 A line chart with several series, labelled directly

A **series** is one set of values plotted together, here one branch's sales. With several series, a **legend** (a key matching colours to names) is the usual way of identifying them, but the reader must look away from the data to read it. **Direct labels**, the name written at the end of each line, avoid that (lecture: titles, captions and labels).

In [ ]:
# 🟢 One line per branch, each named at the end of its line
fig, ax = plt.subplots(figsize=(9, 4))

for i, branch in enumerate(["North", "South", "East"]):
    data = sales[sales["branch"] == branch]
    ax.plot(data["month"], data["sales_k"], color=OKABE_ITO[i], linewidth=2)

    last = data.iloc[-1]                          # the last row: December 2024
    ax.text(last["month"], last["sales_k"],       # position of the text, in data coordinates
            "  " + branch,                        # the text; the spaces move it right of the line's end
            color=OKABE_ITO[i],                   # the same colour as the line it labels
            va="center",                          # vertical alignment: centred on the position
            fontweight="bold")

ax.set(title="South has grown fastest since 2022",
       xlabel="month", ylabel="sales (£ thousand)")
plt.show()

**What you should see:** three lines, each with its branch name at the right-hand end in the same colour.

* **[`ax.text(x, y, "text")`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.text.html)** writes text at a position given in the same units as the data (a date on the x-axis and a sales value on the y-axis), so the label sits exactly at the end of the line.
* **`data.iloc[-1]`** selects the last row of the DataFrame by position (`-1` counts from the end).
* **The title states the finding** ("South has grown fastest") rather than only naming the variables, as the Appendix B checklist recommends.

### 2.2 A scatter plot, with transparency and an annotation

In [ ]:
# 🟢 Weekly study hours against exam score, one point per student
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(students["study_hours"], students["exam_score"],
           alpha=0.5,                 # transparency, from 0 (invisible) to 1 (solid)
           color=OKABE_ITO[0],
           edgecolor="none")          # no outline around each point

ax.set(title="More study hours, higher scores", xlabel="study hours per week", ylabel="exam score")

top = students.loc[students["exam_score"].idxmax()]      # the row of the highest-scoring student
ax.annotate("highest score",                              # the text
            xy=(top["study_hours"], top["exam_score"]),   # the point the arrow points to
            xytext=(top["study_hours"] - 12, top["exam_score"] - 3),   # where the text is placed
            arrowprops={"arrowstyle": "->"})              # draw an arrow from the text to the point
plt.show()

**What you should see:** a cloud of 300 points rising from left to right, and an arrow pointing to the highest score.

* **[`ax.scatter(x, y)`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.scatter.html)** draws one marker for each pair of values.
* **`alpha=0.5`** makes each point half-transparent, so where several points overlap the colour becomes darker. This reveals how dense the data are, which solid points would hide (the lecture calls this **overplotting**).
* **[`ax.annotate`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.annotate.html)** adds text with an optional arrow. `xy` is the position of the point being labelled; `xytext` is the position of the text; `arrowprops` describes the arrow. **`idxmax()`** returns the row label of the largest value, so `students.loc[...]` selects that student's row.

### 2.3 A bar chart, sorted and horizontal

A bar's **length** represents its value, so the value axis must **start at zero**; otherwise the bars' lengths misrepresent the differences (lecture: baselines and truncated axes). Sorting the bars makes the ranking easy to read, and **horizontal** bars leave room for long category names.

In [ ]:
# 🟢 Mean exam score by programme, sorted from lowest to highest
means = (students
         .groupby("programme")["exam_score"]    # split the students by programme
         .mean()                                # the mean exam score of each programme
         .sort_values())                        # sort from smallest to largest
print(means.round(1))

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(means.index, means.values, color=OKABE_ITO[4])    # one bar per programme
for i, value in enumerate(means.values):
    ax.text(value + 0.5, i, f"{value:.1f}", va="center")  # write each value just past the end of its bar

ax.set(title="Mean exam score by programme",
       xlabel="mean exam score",
       xlim=(0, 80))                            # the x-axis from 0 to 80: bars start at zero
plt.show()

**What you should see:** the three means printed (Economics about 66.7, Statistics about 70.5, Data Science about 72.6), then three bars in the same order, each with its value written at its end.

* **[`groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)** splits the rows into groups (one per programme); `.mean()` then calculates the mean exam score within each group.
* **[`ax.barh(categories, values)`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.barh.html)** draws horizontal bars. The bars are placed at positions 0, 1, 2 on the y-axis, which is why the loop writes each value at height `i`.
* **`f"{value:.1f}"`** is an f-string that formats the number with one decimal place.
* **`xlim=(0, 80)`** fixes the x-axis range: it guarantees a zero baseline and leaves space for the value labels.

### 2.4 Histogram and box plot: the distribution of one variable

A **histogram** divides the range of values into intervals (**bins**) and draws a bar for the number of observations in each. A **box plot** summarises the same distribution in five numbers: the **box** spans the middle half of the values (from the first to the third quartile), the line inside it is the **median**, the **whiskers** extend to the furthest values within 1.5 times the box's length, and points beyond the whiskers are drawn individually as possible **outliers** (Tukey, 1977).

In [ ]:
# 🟢 The distribution of weekly study hours, two ways
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5), layout="constrained",
                        gridspec_kw={"width_ratios": [2, 1]})   # the first plot twice as wide as the second

axs[0].hist(students["study_hours"],
            bins=25,                    # the number of bars
            color=OKABE_ITO[0],
            edgecolor="white")          # a white outline separates neighbouring bars
axs[0].set(title="Right-skewed: most students study under 15 hours",
           xlabel="study hours per week", ylabel="number of students")

axs[1].boxplot(students["study_hours"],
               vert=False,              # horizontal, so it lines up with the histogram's x-axis
               widths=0.5)              # the thickness of the box
axs[1].set(title="Box plot", xlabel="study hours per week", yticks=[])   # yticks=[]: no ticks on the y-axis
plt.show()

**What you should see:** a histogram with a long tail to the right (a **right-skewed** distribution: most values are small, a few are large), and a box plot of the same data whose separate points on the right are the students who study unusually long hours.

**`gridspec_kw={"width_ratios": [2, 1]}`** passes settings to the grid that positions the plots ([`GridSpec`](https://matplotlib.org/stable/api/_as_gen/matplotlib.gridspec.GridSpec.html)): here, the first column is twice as wide as the second.

Links: [`ax.hist`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.hist.html), [`ax.boxplot`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.boxplot.html).

### 2.5 Reference lines, gridlines and ticks

* **[`ax.axhline(y)`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.axhline.html)** draws a horizontal line across the whole plot at height `y`; [`ax.axvline(x)`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.axvline.html) draws a vertical one. A reference line shows the reader what to compare the data with, such as a target or an average.
* **[`ax.grid`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.grid.html)** draws gridlines; `axis="y"` limits them to horizontal lines, and a light colour keeps them in the background.
* **[`ax.set_yticks`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.set_yticks.html)** chooses the exact positions of the tick marks on the y-axis.

In [ ]:
# 🟢 East's sales against a target, with light horizontal gridlines
fig, ax = plt.subplots(figsize=(8, 3.5))
east = sales[sales["branch"] == "East"]
ax.plot(east["month"], east["sales_k"], color=OKABE_ITO[2], linewidth=2)

target = 150
ax.axhline(target,
           color="grey",
           linestyle="--",              # a dashed line
           linewidth=1)
ax.text(east["month"].iloc[0], target + 1, "target: £150k", color="grey")   # label the line directly

ax.grid(axis="y", color="#E5E5E5")      # light grey horizontal gridlines only
ax.set_yticks([120, 135, 150, 165])     # tick marks at these four values
ax.set(title="East has rarely reached its target", ylabel="sales (£ thousand)")
plt.show()

**What you should see:** East's monthly sales, a dashed grey line at the £150k target with its label, light horizontal gridlines, and y-axis ticks at 120, 135, 150 and 165 only.

---
## 3. seaborn: describing the chart instead of drawing it

[seaborn](https://seaborn.pydata.org/) is a library for statistical charts, built on top of Matplotlib. The difference is in how you work:

* With **Matplotlib** you draw each element yourself: in section 2.1 we needed a loop to draw one line per branch, and we chose each colour.
* With **seaborn** you give it a **DataFrame** and say which column goes where (the x-axis, the y-axis, the colour). seaborn then splits the data into groups, draws them, chooses the colours, adds a legend and labels the axes with the column names. This style is called **declarative**: you describe the chart, and the library works out how to draw it (lecture: the grammar of graphics).

Because seaborn draws with Matplotlib, every seaborn chart is made of Matplotlib objects, so everything in sections 1 and 2 (`ax.set`, `ax.axhline`, `fig.suptitle`, `fig.savefig`) still works.

### The general form

```python
sns.{function}(data={DataFrame}, x="{column}", y="{column}", hue="{column}", ax={axes})
```

* **`data=`** the DataFrame that holds the columns;
* **`x=`, `y=`** the names of the columns to place on each axis;
* **`hue=`** the name of a column whose values decide the **colour**: one colour per group;
* **`ax=`** the Matplotlib axes to draw on (optional).

### Two kinds of seaborn function

| Kind | Examples | What it draws | What it returns |
|---|---|---|---|
| **axes-level** | [`sns.scatterplot`](https://seaborn.pydata.org/generated/seaborn.scatterplot.html), [`sns.lineplot`](https://seaborn.pydata.org/generated/seaborn.lineplot.html), [`sns.histplot`](https://seaborn.pydata.org/generated/seaborn.histplot.html), [`sns.kdeplot`](https://seaborn.pydata.org/generated/seaborn.kdeplot.html), [`sns.boxplot`](https://seaborn.pydata.org/generated/seaborn.boxplot.html), [`sns.violinplot`](https://seaborn.pydata.org/generated/seaborn.violinplot.html), [`sns.barplot`](https://seaborn.pydata.org/generated/seaborn.barplot.html), [`sns.countplot`](https://seaborn.pydata.org/generated/seaborn.countplot.html), [`sns.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html) | one plot, on an axes you can choose with `ax=` | a Matplotlib axes |
| **figure-level** | [`sns.relplot`](https://seaborn.pydata.org/generated/seaborn.relplot.html), [`sns.displot`](https://seaborn.pydata.org/generated/seaborn.displot.html), [`sns.catplot`](https://seaborn.pydata.org/generated/seaborn.catplot.html) | a whole figure, with one plot per group of a column given by `col=` or `row=` | a seaborn [`FacetGrid`](https://seaborn.pydata.org/generated/seaborn.FacetGrid.html); its figure is `.figure` |

### A theme and a palette

[`sns.set_theme`](https://seaborn.pydata.org/generated/seaborn.set_theme.html) sets the style of all later charts. `style="ticks"` gives a white background with tick marks; `palette="colorblind"` selects seaborn's colour-blind-safe set of colours ([seaborn: choosing colour palettes](https://seaborn.pydata.org/tutorial/color_palettes.html)).

In [ ]:
# 🟢 A seaborn theme with a colour-blind-safe palette, for all the charts that follow
sns.set_theme(style="ticks", palette="colorblind")

### 3.1 A scatter plot with one colour per group: `hue`

[`sns.scatterplot`](https://seaborn.pydata.org/generated/seaborn.scatterplot.html) draws the same kind of chart as `ax.scatter`, but `hue="programme"` colours each point by its programme and adds a legend automatically.

In [ ]:
# 🟢 Create the axes with Matplotlib, draw with seaborn, label with Matplotlib
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.scatterplot(data=students,
                x="study_hours",        # column for the x-axis
                y="exam_score",         # column for the y-axis
                hue="programme",        # column that decides the colour of each point
                alpha=0.7,
                ax=ax)                  # draw on the axes created above
ax.set(title="Study hours and exam score, by programme",
       xlabel="study hours per week", ylabel="exam score")
plt.show()

**What you should see:** the scatter plot of section 2.2, with each programme in its own colour and a legend. This cell shows the pattern used throughout the rest of the lab:

1. create the figure and axes with Matplotlib, `fig, ax = plt.subplots(...)`;
2. draw on it with seaborn, passing `ax=ax`;
3. finish with Matplotlib methods such as `ax.set`.

Without `ax=ax`, seaborn draws on the current axes (section 1), which is less explicit.

### 3.2 A line chart from long-format data: `lineplot`

In [ ]:
# 🟢 One line per branch, without a loop
fig, ax = plt.subplots(figsize=(9, 4))
sns.lineplot(data=sales, x="month", y="sales_k",
             hue="branch",              # one line, in its own colour, for each branch
             linewidth=2, ax=ax)
ax.set(title="Monthly sales by branch", xlabel="month", ylabel="sales (£ thousand)")
plt.show()

**What you should see:** the chart from section 2.1, this time with a legend. Because `sales` is in long format, `hue="branch"` is enough for seaborn to draw one line per branch: compare the length of this code with the loop in 2.1. Link: [`sns.lineplot`](https://seaborn.pydata.org/generated/seaborn.lineplot.html).

### 3.3 Four ways to show a distribution

| Function | Shows |
|---|---|
| [`sns.histplot`](https://seaborn.pydata.org/generated/seaborn.histplot.html) | a histogram: counts of values in bins |
| [`sns.kdeplot`](https://seaborn.pydata.org/generated/seaborn.kdeplot.html) | a **kernel density estimate**: a smooth curve that approximates the shape of the histogram, without the dependence on bin edges |
| [`sns.boxplot`](https://seaborn.pydata.org/generated/seaborn.boxplot.html) | a box plot for each group (section 2.4) |
| [`sns.violinplot`](https://seaborn.pydata.org/generated/seaborn.violinplot.html) | a **violin plot**: the box plot's information combined with a density curve drawn on both sides, which shows the shape of each group's distribution |

In [ ]:
# 🟢 Four views of the exam-score distribution, on a 2 x 2 grid
fig, axs = plt.subplots(2, 2, figsize=(11, 7), layout="constrained")

sns.histplot(data=students, x="exam_score", bins=25, ax=axs[0, 0])          # top left
axs[0, 0].set_title("histplot: counts in bins")

sns.kdeplot(data=students, x="exam_score", hue="programme",
            fill=True,                  # shade the area under each curve
            alpha=0.3,                  # transparent shading, so overlapping curves stay visible
            ax=axs[0, 1])                                                    # top right
axs[0, 1].set_title("kdeplot: a smooth density for each programme")

sns.boxplot(data=students, x="programme", y="exam_score", ax=axs[1, 0])     # bottom left
axs[1, 0].set_title("boxplot: median, quartiles and outliers")

sns.violinplot(data=students, x="programme", y="exam_score",
               inner="quartile",        # draw the median and quartiles as lines inside each violin
               ax=axs[1, 1])                                                 # bottom right
axs[1, 1].set_title("violinplot: the shape of each group")

fig.suptitle("Exam scores: four ways to show a distribution", fontweight="bold")
plt.show()

**What you should see:** a 2 × 2 grid. `axs[0, 0]` is the top-left plot, `axs[0, 1]` the top right, `axs[1, 0]` the bottom left and `axs[1, 1]` the bottom right. All four show scores centred at about 70; the three grouped plots show Data Science a little higher and Economics a little lower. The box and violin plots answer the lecture's question "how do groups differ in distribution?".

### 3.4 Comparing categories: `barplot` and `countplot`

* **`barplot`** draws, for each category, a bar whose height is the **mean** of the y column, with an **error bar**: a black line showing a **95% confidence interval** for that mean. The interval is a range of plausible values for the mean, given the variation in the data; when two groups' intervals overlap a lot, the data do not show a clear difference between them. Showing it follows Appendix B's uncertainty criterion ([seaborn: statistical estimation and error bars](https://seaborn.pydata.org/tutorial/error_bars.html)).
* **`countplot`** draws, for each category, a bar for the **number of rows**.

In [ ]:
# 🟢 Mean score by programme and year, with 95% intervals; and the number of students per programme
fig, axs = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")

sns.barplot(data=students, x="programme", y="exam_score",
            hue="year",                 # two bars per programme: Year 1 and Year 2
            errorbar=("ci", 95),        # error bars show a 95% confidence interval for each mean
            ax=axs[0])
axs[0].set(title="Mean exam score (95% interval)", xlabel="", ylabel="mean exam score")

order = students["programme"].value_counts().index      # programmes from most to fewest students
sns.countplot(data=students, y="programme",
              order=order,              # the order of the bars
              ax=axs[1])
axs[1].set(title="Number of students", xlabel="number of students", ylabel="")
plt.show()

**What you should see:** on the left, two bars (Year 1 and Year 2) for each programme, whose intervals overlap heavily, so the data show no clear difference between the years; on the right, the number of students in each programme, sorted. Links: [`sns.barplot`](https://seaborn.pydata.org/generated/seaborn.barplot.html), [`sns.countplot`](https://seaborn.pydata.org/generated/seaborn.countplot.html).

### 3.5 A correlation matrix: `heatmap`

A **correlation** measures how closely two numeric variables follow a straight-line relationship, from −1 (one rises as the other falls) through 0 (no straight-line relationship) to +1 (both rise together). A **correlation matrix** lists the correlation of every pair of variables. Because correlations have a meaningful midpoint, 0, they need a **diverging** colour scale: one colour for negative values, another for positive values, and white at zero (lecture: three kinds of colour scale; [Matplotlib: choosing colormaps](https://matplotlib.org/stable/users/explain/colors/colormaps.html)).

In [ ]:
# 🟢 Correlations between the four numeric columns
numeric = students[["study_hours", "attendance_pct", "sleep_hours", "exam_score"]]
corr = numeric.corr()                   # a 4 x 4 table of correlations

fig, ax = plt.subplots(figsize=(6, 4.8))
sns.heatmap(corr,
            annot=True,                 # write each value in its cell
            fmt=".2f",                  # with two decimal places
            cmap="RdBu_r",              # a diverging colour map: blue (negative) to red (positive)
            vmin=-1, vmax=1,            # fix the colour scale to the full range of a correlation
            center=0,                   # white at zero
            square=True,                # square cells
            ax=ax)
ax.set_title("Correlations: study hours matter most")
plt.show()

**What you should see:** a 4 × 4 grid with 1.00 along the diagonal (every variable is perfectly correlated with itself), a strong positive correlation between study hours and exam score, and correlations close to zero with sleep (in this synthetic data, sleep was generated independently of the score).

* **`vmin`, `vmax` and `center`** fix the meaning of each colour, so that the same colour always represents the same correlation, whatever the data.
* **`cmap="RdBu_r"`** is Matplotlib's red–blue colour map, reversed (`_r`) so that positive values are red.
* The **colour bar** at the right is drawn by seaborn in a separate, narrow axes of the same figure.

Link: [`sns.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html).

### 3.6 Figure-level functions: one plot per group in one call

The figure-level functions [`sns.relplot`](https://seaborn.pydata.org/generated/seaborn.relplot.html) (relationships), [`sns.displot`](https://seaborn.pydata.org/generated/seaborn.displot.html) (distributions) and [`sns.catplot`](https://seaborn.pydata.org/generated/seaborn.catplot.html) (categories) create **a whole figure** containing one plot for each value of the column given as `col=` (in a row) or `row=` (in a column). The plots share their axes. This is the lecture's **small multiples**, which seaborn calls **faceting**.

They return a seaborn **[`FacetGrid`](https://seaborn.pydata.org/generated/seaborn.FacetGrid.html)**, which holds the figure (`g.figure`) and its axes (`g.axes`), and has its own methods for labelling all the plots at once.

In [ ]:
# 🟢 One scatter plot per programme, all on the same axes ranges
g = sns.relplot(data=students, x="study_hours", y="exam_score",
                col="programme",        # one plot for each programme, side by side
                hue="year",
                height=3.5,             # the height of each plot, in inches
                aspect=1,               # width = aspect x height
                alpha=0.7)
g.set_axis_labels("study hours per week", "exam score")      # the axis labels of every plot
g.set_titles("{col_name}")                                   # each plot titled with its programme
g.figure.suptitle("Study hours and exam score, by programme and year",
                  y=1.04,                # raise the title slightly above the plots
                  fontweight="bold")
plt.show()

print(type(g).__name__, "containing", len(g.axes.flat), "plots")

**What you should see:** three plots side by side, one per programme, with the same axis ranges, and the printout `FacetGrid containing 3 plots`.

**`"{col_name}"`** in `set_titles` is a **template**: seaborn replaces `{col_name}` with each plot's value of the `col` column, so the titles become "Data Science", "Economics" and "Statistics".

In [ ]:
# 🟢 displot: the distribution of exam scores, one plot per year
g = sns.displot(data=students, x="exam_score",
                col="year",             # one plot per year
                hue="programme",
                kind="kde",             # a smooth density curve instead of a histogram
                fill=True, alpha=0.3,
                height=3.2, aspect=1.3)
g.set_axis_labels("exam score", "density")
plt.show()

---
## 4. Combining seaborn and Matplotlib

Since seaborn draws on Matplotlib axes, one figure can mix both libraries: seaborn for the charts, and Matplotlib for anything seaborn does not provide, such as reference lines, annotations and an overall title. The example below also introduces [`sns.regplot`](https://seaborn.pydata.org/generated/seaborn.regplot.html), which draws a scatter plot together with a **fitted regression line** (the straight line that best fits the points) and a shaded band showing its 95% confidence interval.

In [ ]:
# 🟢 A two-plot summary: seaborn charts, finished with Matplotlib
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2), layout="constrained")

# Left: exam scores by programme, with a pass mark
sns.boxplot(data=students, x="programme", y="exam_score",
            color="#D6E6F2",            # one pale colour: the programmes are already named on the axis
            ax=axs[0])
pass_mark = 50
axs[0].axhline(pass_mark, color=OKABE_ITO[5], linestyle="--")                  # Matplotlib
axs[0].text(2.45, pass_mark + 1.5, "pass mark", color=OKABE_ITO[5], ha="right")   # ha: horizontal alignment
axs[0].set(title="Scores by programme", xlabel="", ylabel="exam score")

# Right: attendance against exam score, with a fitted straight line
sns.regplot(data=students, x="attendance_pct", y="exam_score",
            scatter_kws={"alpha": 0.4},  # settings passed on to the scatter points
            ax=axs[1])
axs[1].set(title="Attendance and exam score", xlabel="attendance (%)", ylabel="exam score")

fig.suptitle("Synthetic students: what goes with a higher score?", fontweight="bold")
plt.show()

**What you should see:** on the left, a box plot for each programme with a dashed pass-mark line; on the right, attendance against exam score with a fitted line rising gently and a shaded confidence band.

**`scatter_kws`** (short for "scatter keyword arguments") is a dictionary of settings that `regplot` passes on to the Matplotlib function that draws the points.

### Which object should I use?

| I want to ... | Use | Example |
|---|---|---|
| draw data | the axes | `ax.plot(...)`, `ax.bar(...)`, or `sns.scatterplot(..., ax=ax)` |
| set a plot's title, labels, limits or ticks | the axes | `ax.set(title=..., xlabel=..., ylim=...)` |
| add a reference line, text or an arrow | the axes | `ax.axhline(...)`, `ax.text(...)`, `ax.annotate(...)` |
| add an overall title, or space the plots | the figure | `fig.suptitle(...)`, `plt.subplots(..., layout="constrained")` |
| save to a file | the figure | `fig.savefig("name.png", dpi=200, bbox_inches="tight")` |
| make one plot per group | a seaborn figure-level function | `g = sns.relplot(..., col="...")`, then `g.figure` |

---
## 5. Exercises: reproduce the chart

Each exercise shows a **target chart**, says which data and chart type to use, and gives the exact titles and labels. Write code that reproduces the chart as closely as you can: the same chart type, data, arrangement of plots, titles, labels and annotations. The colours do not need to match exactly.

**For every exercise:**

* Start with `fig, ax = plt.subplots(...)`, or `fig, axs = plt.subplots(...)` for several plots. If you use a seaborn figure-level function, store its result in `g` and then write `fig = g.figure`.
* Make sure the variable **`fig`** holds your figure at the end of the cell: the check cell reads it.
* Then run the check cell. It tests the **structure** of your figure: the number of plots, the titles and the axis labels. It cannot judge how the chart looks, so compare that with the target picture yourself.

The exercises become harder as you go. Each one lists the sections of this notebook to look back at.

In [ ]:
# 🟢 Run once: the check function used after each exercise
def check_figure(fig, n_axes=None, titles=None, xlabels=None, ylabels=None, suptitle=None, min_artists=None):
    """Compare the structure of your figure with the target: the number of plots, the titles and the labels.
    It cannot judge how the chart looks: compare that with the target picture yourself."""
    if fig is None:
        print("❌ No figure yet: create one with fig, ax = plt.subplots(...) in the exercise cell.")
        return
    # A colour bar is drawn in an extra axes; leave it out of the count
    axes = [a for a in fig.axes if a.get_label() != "<colorbar>"]
    results = []
    if n_axes is not None:
        results.append((len(axes) == n_axes, f"{n_axes} plot(s) in the figure (found {len(axes)})"))
    for name, wanted, found in [("title", titles, [a.get_title() for a in axes]),
                                ("x-axis label", xlabels, [a.get_xlabel() for a in axes]),
                                ("y-axis label", ylabels, [a.get_ylabel() for a in axes])]:
        if wanted is not None:
            for i, w in enumerate(wanted):
                got = found[i] if i < len(found) else ""
                results.append((w.lower() in got.lower(), f"plot {i + 1}: {name} contains '{w}' (found '{got}')"))
    if suptitle is not None:
        got = fig._suptitle.get_text() if fig._suptitle else ""
        results.append((suptitle.lower() in got.lower(), f"figure title contains '{suptitle}' (found '{got}')"))
    if min_artists is not None:
        drawn = sum(len(a.lines) + len(a.patches) + len(a.collections) for a in axes)
        results.append((drawn >= min_artists, f"at least {min_artists} drawn elements (found {drawn})"))
    for ok, message in results:
        print("✅" if ok else "❌", message)

### Exercise 1: a line chart with a legend

**Task.** Plot the monthly sales of **all three branches** as three lines on **one plot**, with a **legend**. You may use Matplotlib (`ax.plot` in a loop, giving each line `label=branch`, then `ax.legend()`) or seaborn (`sns.lineplot` with `hue`).

**Details.** Figure size about 9 × 4 inches; title `Monthly sales by branch`; x-axis label `month`; y-axis label `sales (£ thousand)`; legend title `branch`. *Look back at:* 2.1 and 3.2.

**Target chart:**

<img src="https://raw.githubusercontent.com/LeonardooAlves/LSE-ST445/4c9692135e5cd869153a3d84f6fb107b8dc647f3/Week%207/img/e1.png" width="500">

In [ ]:
# 🔵 Your code here. At the end, fig must hold your figure.
fig = None

In [ ]:
# Check
check_figure(fig, n_axes=1, titles=["Monthly sales by branch"], xlabels=["month"], ylabels=["sales"], min_artists=3)

### Exercise 2: a sorted horizontal bar chart, with one bar highlighted

**Task.** Show the **median** exam score of each programme as **horizontal bars, sorted** from lowest to highest, with each value written at the end of its bar. Colour the **highest** bar blue and the others grey: highlighting one element draws the reader's eye to it (lecture: pre-attentive attributes).

**Details.** Title `Data Science has the highest median score`; x-axis label `median exam score`; x-axis from 0 to 80. *Hint:* `students.groupby("programme")["exam_score"].median().sort_values()`; `ax.barh` accepts a list of colours, one per bar, through `color=`. *Look back at:* 2.3.

**Target chart:**

<img src="https://raw.githubusercontent.com/LeonardooAlves/LSE-ST445/4c9692135e5cd869153a3d84f6fb107b8dc647f3/Week%207/img/e2.png" width="500">

In [ ]:
# 🔵 Your code here. At the end, fig must hold your figure.
fig = None

In [ ]:
# Check
check_figure(fig, n_axes=1, titles=["highest median"], xlabels=["median exam score"], min_artists=3)

### Exercise 3: two plots, a histogram and a box plot

**Task.** Make **one figure with two plots side by side**. Left: a **histogram** of `attendance_pct` with 20 bins. Right: a **box plot** of `attendance_pct` for each `programme`, drawn with `sns.boxplot`. Add an **overall title** above both plots, and use constrained layout.

**Details.** Overall title `Attendance: distribution and differences by programme`; left title `Attendance`, x-axis label `attendance (%)`; right title `By programme`, x-axis label `programme`. *Look back at:* 1 (several plots), 2.4 and 3.3.

**Target chart:**

<img src="https://raw.githubusercontent.com/LeonardooAlves/LSE-ST445/4c9692135e5cd869153a3d84f6fb107b8dc647f3/Week%207/img/e3.png" width="500">

In [ ]:
# 🔵 Your code here. At the end, fig must hold your figure.
fig = None

In [ ]:
# Check
check_figure(fig, n_axes=2, titles=["Attendance", "By programme"], xlabels=["attendance", "programme"], suptitle="Attendance: distribution")

### Exercise 4: a scatter plot with colour groups and a reference line

**Task.** Plot `sleep_hours` (x-axis) against `exam_score` (y-axis) with `sns.scatterplot`, coloured by `year`, with partly transparent points. Add a **dashed vertical line at 7 hours** and the text `7 hours` beside it.

**Details.** Title `Sleep and exam score: no clear relationship`; x-axis label `sleep (hours per night)`; y-axis label `exam score`. *Hint:* `ax.axvline(7, ...)` and `ax.text(...)`. *Look back at:* 2.5 and 3.1.

**Target chart:**

<img src="https://raw.githubusercontent.com/LeonardooAlves/LSE-ST445/4c9692135e5cd869153a3d84f6fb107b8dc647f3/Week%207/img/e4.png" width="500">

In [ ]:
# 🔵 Your code here. At the end, fig must hold your figure.
fig = None

In [ ]:
# Check
check_figure(fig, n_axes=1, titles=["no clear relationship"], xlabels=["sleep"], ylabels=["exam score"], min_artists=2)

### Exercise 5: a grouped bar chart with confidence intervals

**Task.** Use `sns.barplot` to show the **mean weekly study hours** of each **programme**, with two bars per programme, one for each **year** (`hue`), and **95% confidence intervals**.

**Details.** Title `Mean weekly study hours (95% interval)`; no x-axis label; y-axis label `mean study hours per week`. *Look back at:* 3.4.

**Target chart:**

<img src="https://raw.githubusercontent.com/LeonardooAlves/LSE-ST445/4c9692135e5cd869153a3d84f6fb107b8dc647f3/Week%207/img/e5.png" width="500">

In [ ]:
# 🔵 Your code here. At the end, fig must hold your figure.
fig = None

In [ ]:
# Check
check_figure(fig, n_axes=1, titles=["Mean weekly study hours"], ylabels=["mean study hours"], min_artists=6)

### Exercise 6: small multiples with a figure-level function

**Task.** Use `sns.displot` to draw a **histogram of exam scores** for each programme, with **one plot per programme** (`col=`), in alphabetical order, sharing the same axes.

**Details.** Plots in the order `col_order=["Data Science", "Economics", "Statistics"]`, each titled with its programme name only; x-axis label `exam score`; y-axis label `number of students`; overall title `Exam scores by programme`. *Remember:* `g = sns.displot(...)`, then `fig = g.figure`. *Look back at:* 3.6.

**Target chart:**

<img src="https://raw.githubusercontent.com/LeonardooAlves/LSE-ST445/4c9692135e5cd869153a3d84f6fb107b8dc647f3/Week%207/img/e6.png" width="500">

In [ ]:
# 🔵 Your code here. At the end, fig must hold your figure.
fig = None

In [ ]:
# Check
check_figure(fig, n_axes=3, titles=["Data Science", "Economics", "Statistics"], xlabels=["exam score"] * 3, suptitle="Exam scores by programme")

### Exercise 7: a correlation heatmap with a diverging palette

**Task.** Draw the **correlation matrix** of `study_hours`, `attendance_pct` and `exam_score` as a heatmap, with each value written in its cell to 2 decimal places, using a **diverging** colour map centred on **zero** and fixed from −1 to 1.

**Details.** Title `Correlations between three variables`. The colour bar is drawn in an extra axes, which the check ignores. *Look back at:* 3.5.

**Target chart:**

<img src="https://raw.githubusercontent.com/LeonardooAlves/LSE-ST445/4c9692135e5cd869153a3d84f6fb107b8dc647f3/Week%207/img/e7.png" width="500">

In [ ]:
# 🔵 Your code here. At the end, fig must hold your figure.
fig = None

In [ ]:
# Check
check_figure(fig, n_axes=1, titles=["Correlations between three variables"])

### Exercise 8 (challenge): a 2 × 2 summary figure

**Task.** Build a figure with **four plots in a 2 × 2 grid**, combining what you have learned: **top left**, the South branch's monthly sales as a line, with a dashed horizontal line at its **mean**; **top right**, a histogram of the South branch's monthly sales; **bottom left**, a box plot of exam score for each year (seaborn); **bottom right**, a `sns.regplot` of study hours against exam score. Add an overall title, use constrained layout, and **save** the figure as `dashboard.png`.

**Details.** Titles: top left `South: monthly sales`; top right `South: distribution`; bottom left `Score by year`; bottom right `Study hours and score`. Overall title `Lab 1 dashboard`. *Look back at:* 1, 2.5, 3.3 and 4.

**Target chart:**

<img src="https://raw.githubusercontent.com/LeonardooAlves/LSE-ST445/4c9692135e5cd869153a3d84f6fb107b8dc647f3/Week%207/img/e8.png" width="500">

In [ ]:
# 🔵 Your code here. At the end, fig must hold your figure.
fig = None

In [ ]:
# Check
check_figure(fig, n_axes=4, titles=["South: monthly sales", "South: distribution", "Score by year", "Study hours and score"], suptitle="Lab 1 dashboard")
import os
if fig is not None:
    print("✅" if os.path.exists("dashboard.png") else "❌", "dashboard.png saved")

---
## Summary

* A **figure** is the whole canvas; each **axes** is one plot inside it. Draw and label on the axes (`ax`); set the overall title, the layout and saving on the figure (`fig`).
* `plt.subplots(nrows, ncols, sharex=..., sharey=..., layout="constrained")` creates a grid of plots; select one with `axs[i]` or `axs[row, column]`. Constrained layout prevents labels from overlapping.
* seaborn draws from a DataFrame given the column names for `x=`, `y=` and `hue=`. Axes-level functions accept `ax=`; figure-level functions (`relplot`, `displot`, `catplot`) make one plot per group with `col=` and `row=`.
* Finish any chart with Matplotlib: `ax.set`, `ax.axhline`, `ax.text`, `ax.annotate`, `fig.suptitle`, `fig.savefig`.
* The lecture's principles, in code: titles that state the finding, direct labels, bars that start at zero, a colour-blind-safe palette, a diverging palette for correlations, shared axes for small multiples, and intervals that show uncertainty.

**Next, Lab 2:** a real dataset, prepared in one cell, then the full eight-step EDA protocol, with every insight stated as a claim, its evidence and its limits.

### Further reading

* Matplotlib: [Quick start guide](https://matplotlib.org/stable/users/explain/quick_start.html); [Anatomy of a figure](https://matplotlib.org/stable/gallery/showcase/anatomy.html); [Constrained layout guide](https://matplotlib.org/stable/users/explain/axes/constrainedlayout_guide.html); [Customising Matplotlib with rcParams](https://matplotlib.org/stable/users/explain/customizing.html); [Choosing colormaps](https://matplotlib.org/stable/users/explain/colors/colormaps.html); [Gallery](https://matplotlib.org/stable/gallery/index.html).
* seaborn: [An introduction to seaborn](https://seaborn.pydata.org/tutorial/introduction.html); [Overview of plotting functions](https://seaborn.pydata.org/tutorial/function_overview.html); [Choosing colour palettes](https://seaborn.pydata.org/tutorial/color_palettes.html); [Statistical estimation and error bars](https://seaborn.pydata.org/tutorial/error_bars.html).
* Okabe, M. and Ito, K. (2008), [Color Universal Design](https://jfly.uni-koeln.de/color/); Tukey, J. W. (1977), *Exploratory Data Analysis*, Addison-Wesley (the box plot); Waskom, M. L. (2021), "seaborn: statistical data visualization", *Journal of Open Source Software*, 6(60), 3021.